# 04 - Blob Storage indexer and enrichment skills (C#)

## Required permissions

- **You:** **Search Service Contributor** + **Search Index Data Contributor** on the workshop Search service, and **Storage Blob Data Contributor** on the sample container. Sign in with `az login` in the deployment tenant.
- **Search managed identity:** **Storage Blob Data Reader** on the container and **Cognitive Services OpenAI User** or equivalent on the parent Foundry resource for embeddings. Bicep grants the broader **Cognitive Services User** role.
- **Resources:** an existing Azure AI Search service with semantic ranker enabled, a Blob container, and an Azure OpenAI embedding deployment. No tenant administrator role is required for this lab.

**Resource prefix:** set `WORKSHOP_PREFIX` in your `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Your index, data source, skillset and indexer use `alice01-blob-*`; uploaded blobs use `alice01/blob/`. Fields, skills and document IDs are scoped inside your objects. Prefixes prevent naming collisions, not enforce access isolation.

Build the pipeline explicitly: **Blob Storage -> data source -> indexer -> Text Split + embedding + Shaper skills -> chunk/vector index -> keyword, vector, hybrid and semantic-ranked search**. Nothing uploads documents directly into Search: the indexer reads the blobs and executes the skillset. Notebook 06 builds a separate index using client-side chunking, embeddings and push ingestion.

Text Split creates overlapping text chunks; Azure OpenAI generates one embedding per chunk; Shaper preserves provenance. Text Split and Shaper are non-billable utility skills, but embeddings and semantic ranking have their own usage/quota. This lab reuses the deployed `workshop-embedding` deployment (`text-embedding-3-small`, 1,536 dimensions). No OCR or Vision skill is used; scanned PDFs need separate extraction.

Use the Search and Storage resources configured in `.env.azure`. Your user needs Search Service Contributor and Search Index Data Contributor on Search, and Storage Blob Data Contributor on the container. Search's system-assigned managed identity needs Storage Blob Data Reader on that container. The repository's [deployment guide](../infra/README.md) is one way to provision these resources and role assignments; if using existing resources, verify the permissions yourself. User credentials upload the source files; Search's identity reads them.

Search's identity also needs embedding inference on the parent Foundry resource: Cognitive Services OpenAI User is sufficient. The repository's Bicep grants the broader Cognitive Services User role, which includes this access. No model API key is used. Semantic ranker must be enabled on Search; confirm regional availability and quota before running the lab.

## 1. Configure your lab

Run `az login --tenant <AZURE_TENANT_ID>` in PowerShell and select the optional legacy .NET (C#) / Polyglot kernel. Run the kernel from the `c-sharp-notebooks` directory. Settings load from `..\.env`, then `..\.env.azure`; file values override existing environment variables, and `.env.azure` wins for duplicate settings. Set `WORKSHOP_PREFIX` in your untracked `.env`. Rerun Configuration after changing settings.

Configure `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`, `AZURE_STORAGE_RESOURCE_ID`, `AZURE_STORAGE_BLOB_ENDPOINT`, `AZURE_STORAGE_CONTAINER_NAME`, `AZURE_AI_ACCOUNT_NAME` and `AZURE_AI_EMBEDDING_DEPLOYMENT_NAME` for your existing resources. The default embedding model is `text-embedding-3-small` with 1,536 dimensions; the model, skill and index must agree. Sample documents are read from `..\data\search`.

Every REST request is visible. Token providers refresh credentials without hiding the request, URL, headers or JSON payload.

This notebook is self-contained: inline BCL helpers and visible REST payloads, no `#!import`, console-project source loading, NuGet or live setup cell. Configuration validates public Azure endpoints and the tenant GUID before writes. It runs independently of the console app. The optional legacy kernel is not the supported default; the .NET 10 console lab is available as an alternative.


In [16]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.IO;
using System.Linq;
using System.Net;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Text.RegularExpressions;
using System.Threading;
using System.Threading.Tasks;

// Inline dotenv, validation, token and HTTP helpers: no console project/source imports.
public static class Workshop
{
public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path))
            return;

        foreach (string line in File.ReadAllLines(path))
        {
            string setting = line.Trim();
            if (setting.Length == 0 || setting.StartsWith("#", StringComparison.Ordinal))
                continue;
            if (setting.StartsWith("export ", StringComparison.Ordinal))
                setting = setting.Substring(7).TrimStart();

            int equals = setting.IndexOf('=');
            if (equals <= 0)
                continue;
            string name = setting.Substring(0, equals).Trim();
            string value = setting.Substring(equals + 1).Trim();
            if (value.Length >= 2 &&
                ((value[0] == '"' && value[value.Length - 1] == '"') ||
                 (value[0] == '\'' && value[value.Length - 1] == '\'')))
            {
                value = value.Substring(1, value.Length - 2);
            }
            else
            {
                int comment = value.IndexOf(" #", StringComparison.Ordinal);
                if (comment >= 0)
                    value = value.Substring(0, comment).TrimEnd();
            }
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}
public sealed record SearchSettings(string Root, string Tenant, string SearchEndpoint,
    string StorageEndpoint, string StorageId, string Container, string Account,
    string Deployment, string Model, string Prefix)
{
    public string Index => Prefix + "-blob-chunks";
    public string DataSource => Prefix + "-blob-source";
    public string Skills => Prefix + "-blob-skills";
    public string Indexer => Prefix + "-blob-indexer";
    public string BlobPrefix => Prefix + "/blob/";
    public string EmbeddingEndpoint => $"https://{Account}.openai.azure.com";

    public static SearchSettings Load(string root)
    {
        Workshop.LoadEnvFile(Path.Combine(root, ".env"));
        Workshop.LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException($"Missing setting: {name}");
        var settings = new SearchSettings(Path.GetFullPath(root), Required("AZURE_TENANT_ID"),
            Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'), Required("AZURE_STORAGE_BLOB_ENDPOINT").TrimEnd('/'),
            Required("AZURE_STORAGE_RESOURCE_ID"), Required("AZURE_STORAGE_CONTAINER_NAME"),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"),
            Environment.GetEnvironmentVariable("AZURE_AI_EMBEDDING_MODEL_NAME")?.Trim() is { Length: > 0 } model
                ? model : "text-embedding-3-small", Required("WORKSHOP_PREFIX"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"^[a-z][a-z0-9]{2,19}$"))
            throw new InvalidOperationException("Use a tenant GUID and a 3-20 lowercase letter/digit prefix starting with a letter.");
        ValidateEndpoint(SearchEndpoint, ".search.windows.net");
        ValidateEndpoint(StorageEndpoint, ".blob.core.windows.net");
        if (!Regex.IsMatch(StorageId, @"^/subscriptions/[0-9a-fA-F-]{36}/resourceGroups/[a-zA-Z0-9_.()-]+/providers/Microsoft\.Storage/storageAccounts/[a-z0-9]{3,24}$") ||
            !Regex.IsMatch(Container, @"^[a-z0-9][a-z0-9-]{1,61}[a-z0-9]$") ||
            !Regex.IsMatch(Account, @"^[a-zA-Z0-9][a-zA-Z0-9-]{1,62}$") ||
            !Regex.IsMatch(Deployment, @"^[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}$") ||
            !Regex.IsMatch(Model, @"^[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}$"))
            throw new InvalidOperationException("Invalid Storage resource ID, container, AI account, deployment or model.");
    }

    static void ValidateEndpoint(string value, string suffix)
    {
        if (!Uri.TryCreate(value, UriKind.Absolute, out var uri) || uri.Scheme != "https" ||
            !uri.Host.EndsWith(suffix, StringComparison.OrdinalIgnoreCase) || uri.Host.Length <= suffix.Length ||
            uri.AbsolutePath != "/" || uri.Query != "" || uri.Fragment != "" || uri.UserInfo != "" || !uri.IsDefaultPort)
            throw new InvalidOperationException($"Expected an HTTPS Azure endpoint ending in {suffix}, without a path/query.");
    }
}


string root = Path.GetFullPath("..");
SearchSettings settings = SearchSettings.Load(root);
const int Dimensions = 1536;
const string SemanticConfiguration = "chunk-semantic";
const string apiVersion = "2025-09-01";
const string searchAudience = "https://search.azure.com/";
const string storageAudience = "https://storage.azure.com/";
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(100) };
Func<string, Task<string>> tokenProvider = resource => AzureCliTokenAsync(settings.Tenant, resource);
Func<TimeSpan, Task> pause = duration => Task.Delay(duration);
Func<DateTimeOffset> clock = () => DateTimeOffset.UtcNow;
string? previousRunStart = null;
bool requestedRunAccepted = false;
async Task<string> AzureCliTokenAsync(string tenant, string resource)
    {
        if (!Guid.TryParseExact(tenant, "D", out _) || resource is not (searchAudience or storageAudience))
            throw new InvalidOperationException("Invalid Azure CLI tenant or audience.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true,
        };
        if (OperatingSystem.IsWindows())
            foreach (string arg in new[] { "/d", "/c", "az" })
                start.ArgumentList.Add(arg);
        foreach (string arg in new[] { "account", "get-access-token", "--resource", resource, "--tenant", tenant, "--output", "json" })
            start.ArgumentList.Add(arg);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync();
        Task<string> errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited)
                process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Sign in to the workshop tenant.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Run az login --tenant with AZURE_TENANT_ID.");
        return JsonNode.Parse(await output)?["accessToken"]?.GetValue<string>() is { Length: > 0 } token
            ? token : throw new InvalidOperationException("Azure CLI returned no access token.");
    }
void CheckResponse(HttpResponseMessage response, string text, params int[] expected)
    {
        if (expected.Contains((int)response.StatusCode))
            return;
        string message = $"HTTP {(int)response.StatusCode} {response.ReasonPhrase}; expected {string.Join(", ", expected)}.";
        Console.Error.WriteLine(message);
        Console.Error.WriteLine("Response body: " + (text.Length == 0 ? "(empty)" : text[..Math.Min(8000, text.Length)]));
        foreach (string name in new[] { "x-ms-request-id", "request-id", "retry-after", "x-ms-retry-after-ms" })
            if (response.Headers.TryGetValues(name, out var values))
                Console.Error.WriteLine($"{name}: {string.Join(", ", values)}");
        throw new HttpRequestException(message, null, response.StatusCode);
    }

async Task<JsonObject> SendCheckedAsync(HttpRequestMessage request, params int[] expected)
{
    // Requests and Authorization stay visible in the cells; no automatic HTTP retries.
    using var response = await client.SendAsync(request);
    string text = await response.Content.ReadAsStringAsync();
    CheckResponse(response, text, expected);
    if (response.StatusCode == HttpStatusCode.NotFound)
        return new JsonObject { ["missing"] = true };
    return string.IsNullOrWhiteSpace(text) ? new JsonObject() : JsonNode.Parse(text)!.AsObject();
}
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("Index: " + settings.Index);
Console.WriteLine("Source folder: " + settings.Container + "/" + settings.BlobPrefix);


Search: https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net
Index: dib01-blob-chunks
Source folder: workshop-documents/dib01/blob/


### Check REST responses before continuing

The requests and payloads stay in their lab cells. This small check prints the HTTP error and response body before raising an exception to stop that cell; a failed request is never reported as ready. It also rejects unexpected successful statuses rather than relying only on `raise_for_status()`.

Blob uploads expect `201`. Search create/update calls accept `200`, `201` or `204`; indexer run requests expect `202`; status and search requests expect `200`. The initial indexer existence check also accepts `404` to mean it has not been created yet. Diagnostics do not print authorization headers; response bodies are limited to 8,000 characters. See [Search HTTP status codes](https://learn.microsoft.com/rest/api/searchservice/http-status-codes).

In [17]:
// The setup helper accepts only the statuses explicitly passed by each request cell.
// Unexpected statuses print an 8,000-character body and request/retry IDs, never Authorization.
// Every failed request raises; polling below is intentional status observation, not HTTP retry.


## 2. Upload source documents to Blob Storage

The repository includes two original synthetic text documents. Upload only these files to your prefix; repeated uploads overwrite only the same two blobs. Add your own text-based PDF, DOCX, PPTX, HTML or TXT files to the same folder later using Storage Explorer. This lab's parser extracts text, not image content.

The storage endpoint remains private to authenticated callers (not anonymously readable). No account key or SAS is used.

In [18]:
// Upload only the two original synthetic samples; never push documents directly to Search.
string[] sampleNames = { "onboarding-guide.txt", "travel-policy.txt" };
foreach (string name in sampleNames)
    if (!File.Exists(Path.Combine(root, "data", "search", name)))
        throw new FileNotFoundException("Expected workshop sample: " + name);
foreach (string name in sampleNames)
{
    string blobName = settings.BlobPrefix + name;
    // Escape each path segment while retaining the virtual-folder slashes.
    string storagePath = string.Join("/", blobName.Split('/').Select(Uri.EscapeDataString));
    using var request = new HttpRequestMessage(HttpMethod.Put,
        $"{settings.StorageEndpoint}/{Uri.EscapeDataString(settings.Container)}/{storagePath}");
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(storageAudience));
    request.Headers.Add("x-ms-version", "2023-11-03");
    request.Headers.Add("x-ms-date", DateTimeOffset.UtcNow.ToString("R", CultureInfo.InvariantCulture));
    request.Headers.Add("x-ms-blob-type", "BlockBlob");
    request.Content = new ByteArrayContent(await File.ReadAllBytesAsync(Path.Combine(root, "data", "search", name)));
    request.Content.Headers.ContentType = MediaTypeHeaderValue.Parse("text/plain; charset=utf-8");
    await SendCheckedAsync(request, 201);
    Console.WriteLine("Uploaded: " + blobName);
}

Uploaded: dib01/blob/onboarding-guide.txt
Uploaded: dib01/blob/travel-policy.txt


## 3. Create the Blob data source

`ResourceId=...;` tells Search to authenticate to Storage using its system-assigned managed identity. `container.query` scopes the indexer to your virtual folder, so it does not ingest everyone else's workshop documents. This is a data selection rule, not a security boundary.

We deliberately omit a deletion policy in this introductory lab. Deleting a source blob alone does not promise removal of its indexed chunks. Configure and test a supported deletion-detection policy before using this pattern for content that must be removed automatically.

In [19]:
// ResourceId selects Search's identity, not a key/SAS. query scopes selection, not authorization.
JsonObject dataSource = JsonNode.Parse($$"""
        {
          "name": "{{settings.DataSource}}", "type": "azureblob",
          "credentials": { "connectionString": "ResourceId={{settings.StorageId}};" },
          "container": { "name": "{{settings.Container}}", "query": "{{settings.BlobPrefix}}" }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/datasources/{settings.DataSource}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(dataSource.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.DataSource);

Ready: dib01-blob-source


## 4. Define a chunk index

Each chunk is a Search document. Index projections generate `chunk_id` and `parent_id`; do not manually map either. The complex `source` field holds Shaper provenance, and `title` holds the source filename.

`text_vector` stores 1,536 floats. Its HNSW profile uses cosine similarity and an Azure OpenAI vectorizer for query-time text embeddings. The skill and vectorizer must use the same deployment/model, and the skill dimensions must match this field. The `semantic` configuration prioritizes `title` and `chunk` for reranking. All three are visible in this REST payload.

If upgrading the old keyword-only lab, rerun from the top: reuploading the samples makes the indexer process them again. Adding a vector field alone does not backfill unchanged documents. For an existing idle indexer, rerunning the indexer cell explicitly requests a new run. Changing an existing vector field's dimensions can require recreating your own index; this notebook never deletes it automatically.

### Use an existing index as a starting point

You can inspect indexes in Azure AI Search and use their definitions as a baseline for your own. Open your Search service, select **Indexes**, then open an index. The **Fields** tab shows field types, searchable/filterable settings, complex fields and vector dimensions.

<img src="../docs/ai-search-index-fields.png" alt="Azure AI Search index Fields tab showing chunk fields, provenance and a 1536-dimensional vector" width="900">

Select **Edit JSON** to inspect the complete index definition, including fields, vector profiles and semantic configurations. This helps connect the portal settings to the REST payload in the next cell.

<img src="../docs/ai-search-index-json-editor.png" alt="Azure AI Search index JSON editor showing the index name and field definitions" width="700">

**Adapt the baseline, don't overwrite the original:** use your own `settings.Index`, remove response metadata such as `@odata.etag`, and review field attributes rather than copying every setting unchanged. Update embedding resource/deployment references, vector dimensions and semantic settings for your resources. The screenshots are illustrative; their names and field settings can differ from this notebook's payload. An index definition creates the schema, not the documents; the indexer populates it later.

In [20]:
// Generated chunk/parent IDs, cosine HNSW and query-time vectorizer; skill/model/dimensions agree.
JsonObject index = JsonNode.Parse($$"""
        {
          "name": "{{settings.Index}}",
          "fields": [
            { "name": "chunk_id", "type": "Edm.String", "key": true, "searchable": true, "analyzer": "keyword" },
            { "name": "parent_id", "type": "Edm.String", "filterable": true },
            { "name": "chunk", "type": "Edm.String", "searchable": true },
            { "name": "title", "type": "Edm.String", "searchable": true },
            { "name": "text_vector", "type": "Collection(Edm.Single)", "searchable": true, "retrievable": true,
              "dimensions": {{Dimensions}}, "vectorSearchProfile": "chunk-vector-profile" },
            { "name": "source", "type": "Edm.ComplexType", "fields": [
              { "name": "file_name", "type": "Edm.String", "searchable": true },
              { "name": "url", "type": "Edm.String" }
            ] }
          ],
          "vectorSearch": {
            "algorithms": [ { "name": "chunk-hnsw", "kind": "hnsw", "hnswParameters": { "metric": "cosine" } } ],
            "profiles": [ { "name": "chunk-vector-profile", "algorithm": "chunk-hnsw", "vectorizer": "chunk-vectorizer" } ],
            "vectorizers": [ {
              "name": "chunk-vectorizer", "kind": "azureOpenAI",
              "azureOpenAIParameters": { "resourceUri": "{{settings.EmbeddingEndpoint}}",
                "deploymentId": "{{settings.Deployment}}", "modelName": "{{settings.Model}}" }
            } ]
          },
          "semantic": {
            "defaultConfiguration": "{{SemanticConfiguration}}",
            "configurations": [ {
              "name": "{{SemanticConfiguration}}",
              "prioritizedFields": {
                "titleField": { "fieldName": "title" },
                "prioritizedContentFields": [ { "fieldName": "chunk" } ]
              }
            } ]
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(index.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.Index);

Ready: dib01-blob-chunks


## 5. Define chunking, embedding and provenance skills

The indexer cracks each source file into `/document/content` and metadata. Text Split writes `/document/chunks/*`. The embedding skill runs on each chunk and writes `/document/chunks/*/text_vector`. Shaper writes `/document/provenance`. Index projections send the text, vector, filename and provenance to the chunk index.

**PDF support:** this Blob indexer handles text-based PDFs as well as the sample TXT files. It extracts the PDF's embedded text, then the same split, embedding and projection pipeline indexes it. Upload PDFs to your `<prefix>/blob/` folder and request another indexer run after any current run finishes.

**Scanned or image-only PDFs need OCR**, which this lab does not configure. These skills also do not preserve physical PDF page numbers. See [notebook 05](../python-notebooks/05-blob-content-understanding.ipynb) for Content Understanding extraction with page metadata.

Omitting `apiKey` and `authIdentity` from both the embedding skill and vectorizer selects Search's system-assigned managed identity. An additional `cognitiveServices` skillset attachment is not required for this embedding skill. Embedding calls incur token charges.

**Text Split has two `textSplitMode` options:**

| Mode | How text is split |
| --- | --- |
| `pages` | Size-based chunks with configurable maximum length and overlap. The skill attempts to preserve sentence boundaries; these are **not physical PDF pages**. |
| `sentences` | Individual sentences, using sentence-ending punctuation and language rules. |

This lab uses `pages` with up to **1,000 characters** per chunk and **200-character overlap**. One PDF page can produce several chunks, and a chunk can span PDF pages. These settings make boundaries easy to inspect, but do not provide page-number citations.

Character-based versus token-based chunk length is a separate `unit` setting (`characters` or `azureOpenAITokens`), not another split mode. This payload omits `unit`, so the default is characters. See the [Text Split skill parameters](https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit#skill-parameters).

In [21]:
// pages means size-based character chunks, not PDF pages. No OCR or cognitiveServices attachment.
// Search's system identity embeds each chunk; Shaper preserves provenance; projections generate IDs.
JsonObject skillset = JsonNode.Parse($$"""
        {
          "name": "{{settings.Skills}}",
          "description": "Split and embed blob text; preserve structured source provenance.",
          "skills": [
            {
              "@odata.type": "#Microsoft.Skills.Text.SplitSkill", "name": "split-text", "context": "/document",
              "textSplitMode": "pages", "maximumPageLength": 1000, "pageOverlapLength": 200,
              "inputs": [ { "name": "text", "source": "/document/content" } ],
              "outputs": [ { "name": "textItems", "targetName": "chunks" } ]
            },
            {
              "@odata.type": "#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill", "name": "embed-chunks",
              "context": "/document/chunks/*", "resourceUri": "{{settings.EmbeddingEndpoint}}",
              "deploymentId": "{{settings.Deployment}}", "modelName": "{{settings.Model}}", "dimensions": {{Dimensions}},
              "inputs": [ { "name": "text", "source": "/document/chunks/*" } ],
              "outputs": [ { "name": "embedding", "targetName": "text_vector" } ]
            },
            {
              "@odata.type": "#Microsoft.Skills.Util.ShaperSkill", "name": "shape-provenance", "context": "/document",
              "inputs": [
                { "name": "file_name", "source": "/document/metadata_storage_name" },
                { "name": "url", "source": "/document/metadata_storage_path" }
              ],
              "outputs": [ { "name": "output", "targetName": "provenance" } ]
            }
          ],
          "indexProjections": {
            "selectors": [ {
              "targetIndexName": "{{settings.Index}}", "parentKeyFieldName": "parent_id",
              "sourceContext": "/document/chunks/*",
              "mappings": [
                { "name": "chunk", "source": "/document/chunks/*" },
                { "name": "text_vector", "source": "/document/chunks/*/text_vector" },
                { "name": "title", "source": "/document/metadata_storage_name" },
                { "name": "source", "source": "/document/provenance" }
              ]
            } ],
            "parameters": { "projectionMode": "skipIndexingParentDocuments" }
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/skillsets/{settings.Skills}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(skillset.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
Console.WriteLine("Ready: " + settings.Skills);

Ready: dib01-blob-skills


## 6. Create the indexer

The indexer connects the data source, skillset and target index. Creating it starts its first run automatically. When this cell updates an existing idle indexer, it explicitly requests a new run; an already-running indexer must finish first. The first run can take several minutes while permissions propagate. No schedule is configured, so additional runs are explicit.

`maxFailedItems=0` and `maxFailedItemsPerBatch=0` prevent a partially failed run from being presented as a successful workshop outcome.

In [22]:
// Clear acceptance first: a failed rerun must not reuse an earlier successful request.
requestedRunAccepted = false;
previousRunStart = null;
JsonObject indexerStatus = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Get,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/status?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    indexerStatus = await SendCheckedAsync(request, 200, 404);
}
bool indexerExists = indexerStatus["missing"]?.GetValue<bool>() != true;
JsonNode? previousRun = indexerStatus["lastResult"];
if (previousRun?["status"]?.GetValue<string>() == "inProgress")
    throw new InvalidOperationException("The indexer is already running. Wait before updating it.");
previousRunStart = previousRun?["startTime"]?.GetValue<string>();
JsonObject indexer = JsonNode.Parse($$"""
        {
          "name": "{{settings.Indexer}}", "dataSourceName": "{{settings.DataSource}}",
          "targetIndexName": "{{settings.Index}}", "skillsetName": "{{settings.Skills}}",
          "parameters": {
            "maxFailedItems": 0, "maxFailedItemsPerBatch": 0,
            "configuration": { "dataToExtract": "contentAndMetadata" }
          }
        }
        """)!.AsObject();
using (var request = new HttpRequestMessage(HttpMethod.Put,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(indexer.ToJsonString(), Encoding.UTF8, "application/json");
    var responseBody = await SendCheckedAsync(request, 200, 201, 204);
}
if (indexerExists)
{
    // Updating an idle existing indexer does not substitute for explicitly requesting its run.
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/run?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    var responseBody = await SendCheckedAsync(request, 202);
}
    Console.WriteLine("New run requested for the existing indexer.");
}
// A new indexer starts automatically. No schedule is configured.
requestedRunAccepted = true;
Console.WriteLine("Indexer ready: " + settings.Indexer);


Indexer ready: dib01-blob-indexer


In [23]:
if (!requestedRunAccepted)
    throw new InvalidOperationException("Run the indexer cell first; no accepted request is pending.");
DateTimeOffset deadline = clock().AddMinutes(15);
bool completed = false;
for (int attempt = 0; attempt < 90 && clock() < deadline; attempt++)
{
    JsonObject status = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Get,
    $"{settings.SearchEndpoint}/indexers/{settings.Indexer}/status?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    status = await SendCheckedAsync(request, 200);
}
    JsonNode? lastRun = status["lastResult"];
    string? state = lastRun?["status"]?.GetValue<string>();
    string? start = lastRun?["startTime"]?.GetValue<string>();
    Console.WriteLine($"Run: {state} | processed: {lastRun?["itemsProcessed"]} | failed: {lastRun?["itemsFailed"]}");
    // While queued, the previous success can still be visible. It is not this requested run.
    bool isRequestedRun = !string.IsNullOrEmpty(start) && start != previousRunStart;
    if (status["status"]?.GetValue<string>() == "error" ||
        (isRequestedRun && state is "transientFailure" or "persistentFailure" or "error"))
        throw new InvalidOperationException(status.ToJsonString());
    if (isRequestedRun && state == "success")
    {
        if ((lastRun?["itemsFailed"]?.GetValue<int>() ?? 0) != 0 || lastRun?["errors"] is JsonArray { Count: > 0 })
            throw new InvalidOperationException(lastRun!.ToJsonString());
        if (clock() >= deadline) break;
        Console.WriteLine("Warnings: " + (lastRun?["warnings"]?.ToJsonString() ?? "[]"));
        completed = true;
        requestedRunAccepted = false;
        break;
    }
    if (attempt < 89 && clock() < deadline)
        await pause(TimeSpan.FromSeconds(10));
}
if (!completed)
    throw new TimeoutException("Indexer did not complete within 15 minutes. Inspect its status before retrying.");


Run: success | processed: 2 | failed: 0
Warnings: []


## 7. Inspect chunks and compare search modes

Inspect chunks and their provenance, then verify stored vector lengths without printing thousands of floats. More chunks than source files demonstrates one-to-many projection.

Use the same question and `top=3` for each search. Keyword uses BM25; vector-only uses embedding similarity; hybrid combines keyword and vector ranks using reciprocal rank fusion (RRF). Semantic ranking reranks candidates and can return extractive captions; it does not generate an LLM answer. Compare filenames, chunk IDs and ordering, not raw scores across modes. With these few documents, rankings can be identical.

The text-vector queries below use the index's vectorizer, so Search's identity calls the embedding deployment. `k=50` supplies up to 50 vector candidates for semantic reranking, even though this tiny sample has fewer chunks.

In [24]:
JsonObject inspectBody = new JsonObject
{
    ["search"] = "*", ["count"] = true, ["select"] = "chunk_id,parent_id,chunk,source", ["top"] = 100
};
JsonObject chunksResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(inspectBody.ToJsonString(), Encoding.UTF8, "application/json");
    chunksResult = await SendCheckedAsync(request, 200);
}
if (chunksResult["value"] is not JsonArray { Count: > 0 })
    throw new InvalidOperationException("No chunks were indexed. Inspect indexer errors and the Blob folder prefix.");
Console.WriteLine("Total chunks: " + chunksResult["@odata.count"]);
Console.WriteLine(chunksResult["value"]!.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));


Total chunks: 6
[
  {
    "@search.score": 1.0,
    "chunk_id": "872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1_chunks_2",
    "parent_id": "aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1",
    "chunk": "of personal costs. The fictional organization pays only the documented business portion of eligible expenses.\r\n\r\nReview and escalation\r\nThe manager confirms that expenses relate to the approved trip. The finance reviewer checks receipts, dates, arithmetic and policy exceptions. A rejected reimbursement request should include a reason and the information needed to correct it. Questions about an unusual expense should be raised before travel whenever possible. These rules are deliberately synthetic so that workshop participants can share search results safely.",
    "source": {
      "file_name": "travel-pol

### Check a sample of stored embeddings

The next cell checks that indexed chunks have vectors of the expected size. `search='*'` retrieves up to **three chunks**; this is **not a vector-similarity search**. `select` returns only each chunk's ID and stored `text_vector`.

After requiring HTTP `200`, the cell checks that every sampled vector has **1,536 values** (`Dimensions`). Missing results, missing vectors or wrong-sized vectors raise an error. It prints chunk IDs and vector lengths instead of thousands of numbers.

This is a **sample check**, not verification of every indexed chunk or the quality of its embedding.

In [25]:
// Sample only: this reads stored embeddings; it is not a vector-similarity query.
JsonObject sampleBody = new JsonObject { ["search"] = "*", ["select"] = "chunk_id,text_vector", ["top"] = 3 };
JsonObject sampleResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(sampleBody.ToJsonString(), Encoding.UTF8, "application/json");
    sampleResult = await SendCheckedAsync(request, 200);
}
if (sampleResult["value"] is not JsonArray { Count: > 0 } vectorSamples ||
    vectorSamples.Any(item => item?["text_vector"] is not JsonArray { Count: Dimensions }))
    throw new InvalidOperationException("Missing or wrong-sized vectors. Inspect embedding errors.");
foreach (var item in vectorSamples)
    Console.WriteLine($"{item!["chunk_id"]} | vector dimensions: {Dimensions}");


872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1_chunks_2 | vector dimensions: 1536
872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2Ivb25ib2FyZGluZy1ndWlkZS50eHQ1_chunks_1 | vector dimensions: 1536
872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2Ivb25ib2FyZGluZy1ndWlkZS50eHQ1_chunks_0 | vector dimensions: 1536


### Keyword search (BM25)

In [26]:
string question = "What receipts do I need to claim travel expenses, and when must I submit them?";
JsonObject keywordBody = new JsonObject
{
    ["search"] = question, ["searchFields"] = "chunk",
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3
};
JsonObject keywordResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(keywordBody.ToJsonString(), Encoding.UTF8, "application/json");
    keywordResult = await SendCheckedAsync(request, 200);
}
Console.WriteLine(keywordResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-blob-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 2.053158,
      "chunk_id": "872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1_chunks_0",
      "chunk": "Northwind Workshop - Synthetic Travel Policy\r\n\r\nThis document is fictional workshop material. It contains no real employee records or corporate policy. Use it to explore indexing, chunk boundaries, provenance and keyword retrieval.\r\n\r\nTravel planning\r\nEmployees should discuss the purpose, expected outcomes and estimated cost of a trip with their manager before booking. The fictional Northwind Workshop organization prefers remote meetings when they achieve the same result. A trip request includes destination, travel dates, transport options and a short explanation of the customer or training act

### Vector-only search

There is deliberately no `search` property: this is not a hybrid request.

**The question is embedded at query time.** With `kind='text'`, the request passes `question` to Azure AI Search. The index's configured Azure OpenAI vectorizer generates an embedding for that question, then Search compares it with the chunk vectors already stored in `text_vector`.

The indexer's embedding skill generated the document-chunk embeddings earlier; they are not regenerated for every search. Query-time embedding calls use Search's managed identity and incur embedding usage charges. The hybrid queries below also use this text-to-vector step, alongside keyword retrieval.

In [27]:
// No search property: query-time embedding uses Search's identity, not a user OpenAI call.
JsonObject vectorBody = new JsonObject
{
    ["vectorQueries"] = new JsonArray(new JsonObject
    {
        ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50
    }),
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3
};
JsonObject vectorResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(vectorBody.ToJsonString(), Encoding.UTF8, "application/json");
    vectorResult = await SendCheckedAsync(request, 200);
}
Console.WriteLine(vectorResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-blob-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 0.7198458,
      "chunk_id": "872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1_chunks_1",
      "chunk": "after the trip ends. Attach an itemized receipt for lodging, transport and any individual expense greater than twenty-five fictional credits. A card statement by itself is not an itemized receipt. If a receipt is missing, include a written explanation and request an exception from the finance reviewer. Duplicate claims are not permitted. Personal entertainment and unrelated shopping are not reimbursable.\r\n\r\nHotels and transport\r\nSelect a reasonably priced hotel near the meeting location. Public transit is encouraged when practical. If a rental car is necessary, record why public transit was not su

### Hybrid search (keyword + vector, fused with RRF)

In [28]:
JsonObject hybridBody = new JsonObject
{
    ["search"] = question, ["searchFields"] = "chunk",
    ["vectorQueries"] = new JsonArray(new JsonObject
    {
        ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50
    }),
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3
};
JsonObject hybridResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(hybridBody.ToJsonString(), Encoding.UTF8, "application/json");
    hybridResult = await SendCheckedAsync(request, 200);
}
Console.WriteLine(hybridResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-blob-chunks\u0027)/$metadata#docs(*)",
  "value": [
    {
      "@search.score": 0.03279569745063782,
      "chunk_id": "872459a256c1_aHR0cHM6Ly9zdGlxNmp2a3R2NjNzZnY0ZS5ibG9iLmNvcmUud2luZG93cy5uZXQvd29ya3Nob3AtZG9jdW1lbnRzL2RpYjAxL2Jsb2IvdHJhdmVsLXBvbGljeS50eHQ1_chunks_0",
      "chunk": "Northwind Workshop - Synthetic Travel Policy\r\n\r\nThis document is fictional workshop material. It contains no real employee records or corporate policy. Use it to explore indexing, chunk boundaries, provenance and keyword retrieval.\r\n\r\nTravel planning\r\nEmployees should discuss the purpose, expected outcomes and estimated cost of a trip with their manager before booking. The fictional Northwind Workshop organization prefers remote meetings when they achieve the same result. A trip request includes destination, travel dates, transport options and a short explanation of the customer or t

### Keyword search + semantic ranking

Look for `@search.rerankerScore` and `@search.captions`. A partial semantic response is an error here, not a successful reranking demonstration.

In [29]:
JsonObject semanticBody = new JsonObject
{
    ["search"] = question, ["searchFields"] = "chunk",
    ["queryType"] = "semantic", ["semanticConfiguration"] = SemanticConfiguration, ["captions"] = "extractive",
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3
};
JsonObject semanticResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(semanticBody.ToJsonString(), Encoding.UTF8, "application/json");
    semanticResult = await SendCheckedAsync(request, 200);
}
if (semanticResult["@search.semanticPartialResponseReason"] is not null)
    throw new InvalidOperationException("Semantic ranking did not complete: " + semanticResult.ToJsonString());
Console.WriteLine(semanticResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-blob-chunks\u0027)/$metadata#docs(*)",
  "@search.answers": [],
  "value": [
    {
      "@search.score": 2.053158,
      "@search.rerankerScore": 2.534362316131592,
      "@search.captions": [
        {
          "text": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and receipts Submit travel reimbursement requests within thirty calendar days after the trip ends. Attach an.",
          "highlights": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement\u003Cem\u003E and receipts Submit travel reimbursement request

### Hybrid search + semantic ranking

Keyword and vector retrieval first produce candidates; semantic ranking reranks the fused results.

In [30]:
JsonObject semanticHybridBody = new JsonObject
{
    ["search"] = question, ["searchFields"] = "chunk",
    ["vectorQueries"] = new JsonArray(new JsonObject
    {
        ["kind"] = "text", ["text"] = question, ["fields"] = "text_vector", ["k"] = 50
    }),
    ["queryType"] = "semantic", ["semanticConfiguration"] = SemanticConfiguration, ["captions"] = "extractive",
    ["select"] = "chunk_id,title,chunk,source", ["top"] = 3
};
JsonObject semanticHybridResult = new JsonObject();
using (var request = new HttpRequestMessage(HttpMethod.Post,
    $"{settings.SearchEndpoint}/indexes/{settings.Index}/docs/search?api-version={apiVersion}"))
{
    request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", await tokenProvider(searchAudience));
    request.Content = new StringContent(semanticHybridBody.ToJsonString(), Encoding.UTF8, "application/json");
    semanticHybridResult = await SendCheckedAsync(request, 200);
}
if (semanticHybridResult["@search.semanticPartialResponseReason"] is not null)
    throw new InvalidOperationException("Semantic ranking did not complete: " + semanticHybridResult.ToJsonString());
Console.WriteLine(semanticHybridResult.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

{
  "@odata.context": "https://iq-workshop-search-eastus-suy43pebejwnk.search.windows.net/indexes(\u0027dib01-blob-chunks\u0027)/$metadata#docs(*)",
  "@search.answers": [],
  "value": [
    {
      "@search.score": 0.03279569745063782,
      "@search.rerankerScore": 2.534362316131592,
      "@search.captions": [
        {
          "text": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement and receipts Submit travel reimbursement requests within thirty calendar days after the trip ends. Attach an.",
          "highlights": "Northwind Workshop - Synthetic Travel Policy  This document is fictional workshop material. It contains no real employee records or corpora... Manager approval should be recorded before any nonrefundable booking. Travel reimbursement\u003Cem\u003E and receipts Submit travel reimbursem

## References

- [Blob indexer](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-storage)
- [Managed identity access to Storage](https://learn.microsoft.com/azure/search/search-how-to-managed-identities-storage)
- [Text Split skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit)
- [Shaper skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-shaper)
- [Azure OpenAI embedding skill](https://learn.microsoft.com/azure/search/cognitive-search-skill-azure-openai-embedding)
- [Azure OpenAI vectorizer](https://learn.microsoft.com/azure/search/vector-search-vectorizer-azure-open-ai)
- [Hybrid search](https://learn.microsoft.com/azure/search/hybrid-search-overview)
- [Semantic ranking](https://learn.microsoft.com/azure/search/semantic-search-overview)
- [REST pipeline reference](https://github.com/memasanz/mm-ai-search/blob/main/000_CreateIndex.ipynb)
- [Index projections](https://learn.microsoft.com/azure/search/search-how-to-define-index-projections)
- [Blob deletion detection](https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-changed-deleted)

Next: [notebook 05](../python-notebooks/05-blob-content-understanding.ipynb) uses a separate Blob indexer with Content Understanding to extract and chunk a PDF with page metadata. [Notebook 06](../python-notebooks/06-push-chunks-vectors.ipynb) instead chunks TXT files and generates embeddings in Python, then pushes them to a separate index.